In [ ]:
import requests
import os
import aiohttp
import json
import time
url=os.environ.get("databricksURL")
wid=os.environ.get("databricksWID")
apiKey=os.environ.get("databricksAPI")
print(url)
print(wid)

url = 'https://'+url+'.cloud.databricks.com/api/2.0/sql/statements'
myobj = {
  "warehouse_id":wid,
  "catalog": "bgsage",
  "schema": "default",
  "statement": "SELECT * from bgsage where bgsage.embedding IS NULL"
}

while True:
  async with aiohttp.ClientSession() as session:
      async with session.post(url, json=myobj,headers={"Authorization":"Bearer "+apiKey}) as res:
        result = await res.json()  # or response.text() for text
        print(result)
        state=result["status"]["state"]
        if state == "SUCCEEDED":
            break
        elif state == "PENDING":
            print(f"Status: {state}, waiting...")
            await time.sleep(.25)  # Wait 2 seconds before checking again
            
        else:
            print("Operation failed!")
            raise Exception(f"Operation failed: {state}")
  #resX=json.loads(x.text) 
  print("SS")
  print(result["result"]["data_array"])

dbc-b18aa4ad-0c6f
e846941594bb6d72
{'statement_id': '01f102d5-d34f-12e9-a9f2-c5faf3697acd', 'status': {'state': 'SUCCEEDED'}, 'manifest': {'format': 'JSON_ARRAY', 'schema': {'column_count': 5, 'columns': [{'name': 'id', 'type_text': 'BIGINT', 'type_name': 'LONG', 'position': 0}, {'name': 'text_content', 'type_text': 'STRING', 'type_name': 'STRING', 'position': 1}, {'name': 'embedding', 'type_text': 'ARRAY<FLOAT>', 'type_name': 'ARRAY', 'position': 2}, {'name': 'page_number', 'type_text': 'INT', 'type_name': 'INT', 'position': 3}, {'name': 'document_source', 'type_text': 'STRING', 'type_name': 'STRING', 'position': 4}]}, 'total_chunk_count': 1, 'chunks': [{'chunk_index': 0, 'row_offset': 0, 'row_count': 2}], 'total_row_count': 2, 'truncated': False}, 'result': {'chunk_index': 0, 'row_offset': 0, 'row_count': 2, 'data_array': [['1', 'In the Crawl Mode, players play through a chosen Scenario. The end goal is determined by the Scenario the players choose.', None, '4', 'Middara'], ['2', 'Sk

In [23]:
from sentence_transformers import SentenceTransformer
data=result["result"]["data_array"]
sentences = ["This is an example sentence", "Each sentence is converted"]

model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
#for d in data:
for d in data:
    embeddings = model.encode(d[1])
    embString="".join(f",{e:.25g}" for e in embeddings)[1:]
    myobj = {
        "warehouse_id":wid,
        "catalog": "bgsage",
        "schema": "default",
        "statement": "UPDATE bgsage SET bgsage.embedding=CAST(array("+embString+") AS ARRAY<FLOAT>) WHERE bgsage.id="+str(d[0])
        }
    print("".join(f"{e:.25g}," for e in embeddings))
    x = requests.post(url, json = myobj,headers={"Authorization":"Bearer "+apiKey})
    resX=json.loads(x.text) 
    print(resX)

-0.06633235514163970947265625,0.02696579322218894958496094,0.02946047857403755187988281,0.02167874388396739959716797,0.05530666932463645935058594,0.03052662312984466552734375,-0.007551644928753376007080078,-0.03680731728672981262207031,0.009616703726351261138916016,0.08863155543804168701171875,0.02280279248952865600585938,-0.007838066667318344116210938,0.003891363972797989845275879,0.01757352054119110107421875,-0.002214659005403518676757812,-0.003044554032385349273681641,-0.05100635066628456115722656,0.02042132057249546051025391,0.02863796986639499664306641,-0.0497185289859771728515625,0.09237404912710189819335938,-0.08371315896511077880859375,-0.005544807761907577514648438,-0.12622487545013427734375,-0.05875198915600776672363281,0.04438408464193344116210938,-0.03987583890557289123535156,0.0005725545925088226795196533,0.03628180921077728271484375,-0.02643237821757793426513672,0.05470867455005645751953125,0.046839177608489990234375,0.01727366074919700622558594,0.001311126397922635078430